In [1]:
import math

def generate_saddle_plane(filename):
    vertices = []
    faces = []
    vertex_count = 0
    
    # --- 参数设置 ---
    size = 3.0
    res = 40  # 网格密度
    
    # Primitive A: Hyperbolic Paraboloid (Saddle)
    # z = scale * (x^2 - y^2)
    saddle_scale = 0.5 
    
    # Primitive B: Plane
    # z = 0 (Tangent at origin)
    
    # ==========================
    # 1. 生成 Saddle
    # ==========================
    saddle_start = 1
    for i in range(res + 1):
        x = -size + 2 * size * i / res
        for j in range(res + 1):
            y = -size + 2 * size * j / res
            
            # 鞍面方程
            z = saddle_scale * (x**2 - y**2)
            
            vertices.append((x, y, z))
            vertex_count += 1
            
    # Saddle Faces
    for i in range(res):
        for j in range(res):
            row = res + 1
            base = saddle_start + i * row + j
            p0 = base
            p1 = base + 1
            p2 = base + row + 1
            p3 = base + row
            
            faces.append(f"f {p0} {p1} {p2}")
            faces.append(f"f {p0} {p2} {p3}")

    # ==========================
    # 2. 生成 Plane (z=0)
    # ==========================
    plane_start = vertex_count + 1
    # 我们生成一个稍微大一点的平面以便观察
    plane_size = size * 1.1
    
    # 简单的 4 顶点平面 (或者细分网格以获得更好的着色)
    # 这里用细分网格，以便在顶点级别看清相交
    for i in range(res + 1):
        x = -plane_size + 2 * plane_size * i / res
        for j in range(res + 1):
            y = -plane_size + 2 * plane_size * j / res
            z = 0.0
            vertices.append((x, y, z))
            vertex_count += 1
            
    for i in range(res):
        for j in range(res):
            row = res + 1
            base = plane_start + i * row + j
            p0 = base
            p1 = base + 1
            p2 = base + row + 1
            p3 = base + row
            
            faces.append(f"f {p0} {p2} {p1}") # 注意法线方向
            faces.append(f"f {p0} {p3} {p2}")

    # ==========================
    # 写入 OBJ
    # ==========================
    with open(filename, 'w') as f:
        f.write("# Hyperbolic Paraboloid (Saddle) + Tangent Plane\n")
        f.write("# Intersection creates crossed lines with a Flat center\n")
        
        for v in vertices:
            f.write(f"v {v[0]:.4f} {v[1]:.4f} {v[2]:.4f}\n")
            
        f.write("\ng Saddle\n")
        saddle_face_count = res * res * 2
        for i in range(saddle_face_count):
            f.write(faces[i] + "\n")
            
        f.write("\ng Plane\n")
        for i in range(saddle_face_count, len(faces)):
            f.write(faces[i] + "\n")
            
    print(f"Generated '{filename}'. Look for the 'X' shaped intersection at the center.")

if __name__ == "__main__":
    generate_saddle_plane("saddle_intersection.obj")

Generated 'saddle_intersection.obj'. Look for the 'X' shaped intersection at the center.


In [8]:
from build123d import *

# 1. Define dimensions
leg_length = 50.0  # Length of the legs
thickness = 10.0    # Thickness of the wall
width = 20.0       # Extrusion width
angle = 60.0      # Angle between the two planes
fillet_radius = 2.0

# 2. Create the Part
with BuildPart() as model:
    with BuildSketch() as sk:
        # A. Create Leg 1 (Horizontal)
        # Align.MIN puts the corner at (0,0)
        leg1 = Rectangle(leg_length, thickness, align=(Align.MIN, Align.MIN))
        
        # B. Create Leg 2 (Rotated)
        # We create a rectangle and rotate it 120 degrees around the origin
        leg2 = Rectangle(leg_length, thickness, align=(Align.MIN, Align.MIN))
        leg2 = leg2.rotate(Axis.Z, angle)
        
        # C. Combine them (Union)
        add(leg1)
        add(leg2)
        
        # D. Fillet the inner corner
        # The union creates a vertex exactly at (0,0). 
        # We find it by sorting vertices by distance to the origin.
        corner_vertex = sk.vertices().sort_by_distance((0,0))[0]
        fillet(corner_vertex, radius=fillet_radius)

    # 3. Extrude to 3D
    extrude(amount=width)

# 4. Export
model.part.export_stl("fillet_transition.stl")

print(f"Success: Model generated with {angle}-degree angle between patches.")

Success: Model generated with 60.0-degree angle between patches.


C:\Users\ruan\AppData\Local\Temp\ipykernel_29092\832484029.py:36: DeprecationWarning: Use the `export_stl` function instead
  model.part.export_stl("fillet_transition.stl")
